# Proyecto Módulo 6: Predicción de gasto en clientes e-commerce

**Nombre:**  Carolina Tapia Bahamonde
**Curso:**  Ciencia de datos
**Fecha:** 19-03-2026 

---

## 1. Introducción

En el contexto del comercio electrónico, comprender el comportamiento de compra de los clientes es fundamental para la toma de decisiones estratégicas.

El objetivo de este proyecto es desarrollar un modelo de aprendizaje supervisado que permita predecir el monto total de compra de un cliente en función de sus características y comportamiento.

## 2. Definición del problema

El problema se define como una tarea de aprendizaje supervisado de tipo regresión, ya que se busca predecir una variable numérica continua (gasto del cliente).

El pipeline del proyecto incluye:

- Carga y exploración de datos
- Preprocesamiento
- División de datos
- Entrenamiento de modelos
- Evaluación mediante métricas
- Optimización
- Selección del modelo final

## 3. Importación de librerías



Se importan las librerías necesarias para el análisis de datos, modelado y evaluación.

In [1]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.ensemble import GradientBoostingRegressor

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

## 4. Selección del dataset

Se selecciona el dataset **ecommerce_customer_data_large.csv**, ya que contiene datos en su estado original, permitiendo aplicar completamente el pipeline de machine learning.

Trabajar con datos sin procesar permite realizar análisis exploratorio, preprocesamiento y evaluación más realista del modelo.

El dataset basado en ratios no se selecciona debido a que ya contiene transformaciones previas, limitando el análisis.

## 5. Carga de datos


Se carga el dataset de clientes de e-commerce.

In [2]:
df = pd.read_csv("data/ecommerce_customer_data_large.csv")
df.head()

,Customer ID,Purchase Date,Product Category,Product Price,Quantity,Total Purchase Amount,Payment Method,Customer Age,Returns,Customer Name,Age,Gender,Churn
0,44605,2023-05-03 21:30:02,Home,177,1,2427,PayPal,31,1.0,John Rivera,31,Female,0
1,44605,2021-05-16 13:57:44,Electronics,174,3,2448,PayPal,31,1.0,John Rivera,31,Female,0
2,44605,2020-07-13 06:16:57,Books,413,1,2345,Credit Card,31,1.0,John Rivera,31,Female,0
3,44605,2023-01-17 13:14:36,Electronics,396,3,937,Cash,31,0.0,John Rivera,31,Female,0
4,44605,2021-05-01 11:29:27,Books,259,4,2598,PayPal,31,1.0,John Rivera,31,Female,0


## 6. Análisis exploratorio de datos

Se analiza la estructura del dataset, tipos de variables y posibles valores nulos.

In [3]:
df.shape

(250000, 13)

Se observa la cantidad de registros y columnas del dataset.

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 250000 entries, 0 to 249999
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   Customer ID            250000 non-null  int64  
 1   Purchase Date          250000 non-null  str    
 2   Product Category       250000 non-null  str    
 3   Product Price          250000 non-null  int64  
 4   Quantity               250000 non-null  int64  
 5   Total Purchase Amount  250000 non-null  int64  
 6   Payment Method         250000 non-null  str    
 7   Customer Age           250000 non-null  int64  
 8   Returns                202618 non-null  float64
 9   Customer Name          250000 non-null  str    
 10  Age                    250000 non-null  int64  
 11  Gender                 250000 non-null  str    
 12  Churn                  250000 non-null  int64  
dtypes: float64(1), int64(7), str(5)
memory usage: 24.8 MB


Se identifican los tipos de datos de cada variable.

In [5]:
df.describe()

,Customer ID,Product Price,Quantity,Total Purchase Amount,Customer Age,Returns,Age,Churn
count,250000.000000,250000.000000,250000.000000,250000.000000,250000.000000,202618.000000,250000.000000,250000.00000
mean,25017.632092,254.742724,3.004936,2725.385196,43.798276,0.500824,43.798276,0.20052
std,14412.515718,141.738104,1.414737,1442.576095,15.364915,0.500001,15.364915,0.40039
min,1.000000,10.000000,1.000000,100.000000,18.000000,0.000000,18.000000,0.00000
25%,12590.000000,132.000000,2.000000,1476.000000,30.000000,0.000000,30.000000,0.00000
50%,25011.000000,255.000000,3.000000,2725.000000,44.000000,1.000000,44.000000,0.00000
75%,37441.250000,377.000000,4.000000,3975.000000,57.000000,1.000000,57.000000,0.00000
max,50000.000000,500.000000,5.000000,5350.000000,70.000000,1.000000,70.000000,1.00000


Se observa que la variable objetivo "Total Purchase Amount" presenta una amplia variabilidad, lo que la hace adecuada para un problema de regresión.

Las variables numéricas como "Product Price", "Quantity" y "Customer Age" presentan distribuciones coherentes y sin valores extremos evidentes.

La variable "Returns" es de tipo binaria, lo que indica si un producto fue devuelto o no, siendo una variable potencialmente relevante para el modelo.

En general, el dataset presenta buena calidad, con valores consistentes y sin anomalías significativas.

Se analizan estadísticas descriptivas de variables numéricas.

In [6]:
df.isnull().sum()

Customer ID                  0
Purchase Date                0
Product Category             0
Product Price                0
Quantity                     0
Total Purchase Amount        0
Payment Method               0
Customer Age                 0
Returns                  47382
Customer Name                0
Age                          0
Gender                       0
Churn                        0
dtype: int64

Se verifica la existencia de valores nulos.

## 7. Limpieza y definición de variables

En esta etapa se realiza la preparación del dataset para el modelado.

Se ejecutan las siguientes acciones:

- Eliminación de variables irrelevantes (identificadores y nombres)
- Eliminación de variables redundantes
- Tratamiento de valores nulos
- Definición de variable objetivo y variables predictoras

In [7]:
# eliminar columnas irrelevantes
df = df.drop(['Customer ID', 'Customer Name', 'Purchase Date'], axis=1)

# eliminar duplicidad de edad
df = df.drop(['Age'], axis=1)

# tratar valores nulos en Returns
df['Returns'] = df['Returns'].fillna(0)

Se eliminan las variables "Customer ID" y "Customer Name", ya que no aportan valor predictivo al modelo.

La variable "Purchase Date" se elimina debido a que no será transformada en este análisis, evitando complejidad adicional.

Se elimina la variable "Age" debido a que duplica la información de "Customer Age", lo que podría generar redundancia en el modelo.

Los valores nulos en la variable "Returns" se reemplazan por 0, asumiendo que la ausencia de registro implica que no hubo devoluciones.

## Definición de variables

Se define la variable objetivo (target) como el monto total de compra, y las variables restantes como predictoras.

In [8]:
# variable objetivo
y = df['Total Purchase Amount']

# variables predictoras
X = df.drop('Total Purchase Amount', axis=1)

## 8. Codificación de variables categóricas

Se transforman las variables categóricas en variables numéricas mediante One-Hot Encoding.

In [9]:
X = pd.get_dummies(X, drop_first=True)

## 9. Escalamiento de datos

Se aplica estandarización a las variables predictoras con el objetivo de llevarlas a una escala común.

Esto es especialmente importante para modelos basados en distancia, como KNN, ya que variables con diferentes magnitudes pueden sesgar el resultado.

In [10]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

## 10. División de datos

Se divide el dataset en conjuntos de entrenamiento y prueba para evaluar el desempeño del modelo en datos no vistos.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [12]:
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(200000, 11) (50000, 11)
(200000,) (50000,)


Se verifica que la división de los datos se haya realizado correctamente, manteniendo la proporción entre los conjuntos de entrenamiento (80%) y prueba (20%).

Esta separación permite evaluar la capacidad de generalización del modelo, asegurando que el desempeño observado no esté sesgado por los datos utilizados en el entrenamiento.

## 11. Modelo de regresión lineal

Se entrena un modelo de regresión lineal como línea base para evaluar el comportamiento inicial del problema.

In [13]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred_lr = lr.predict(X_test)

## 12. Modelo KNN

Se entrena un modelo KNN para capturar posibles relaciones no lineales en los datos.

In [14]:
knn = KNeighborsRegressor(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred_knn = knn.predict(X_test)

## 13. Evaluación de modelos

Se evalúan los modelos utilizando métricas de regresión:

- MAE (Error absoluto medio)
- RMSE (Raíz del error cuadrático medio)
- R² (Coeficiente de determinación)

Estas métricas permiten analizar el error y la capacidad predictiva del modelo.

In [15]:
def evaluar(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "R2": r2_score(y_true, y_pred)
    }

res_lr = evaluar(y_test, y_pred_lr)
res_knn = evaluar(y_test, y_pred_knn)

res_lr, res_knn

({'MAE': 1245.436450925946,
  'RMSE': np.float64(1438.2719363148092),
  'R2': 0.003027585699719393},
 {'MAE': 1331.1806920000001,
  'RMSE': np.float64(1576.7287811062497),
  'R2': -0.19816083385330452})

Se observa que el modelo de regresión lineal presenta un mejor desempeño en comparación con KNN en todas las métricas evaluadas.

El modelo de regresión lineal obtiene un MAE de 1245.44 y un RMSE de 1438.27, mientras que KNN presenta errores mayores, lo que indica menor precisión en las predicciones.

En cuanto al coeficiente de determinación (R²), la regresión lineal alcanza un valor cercano a 0 (0.003), lo que indica que el modelo explica muy poca variabilidad de la variable objetivo. Sin embargo, el modelo KNN presenta un R² negativo (-0.198), lo que indica que su desempeño es peor que simplemente predecir el promedio.

Estos resultados sugieren que ninguno de los modelos logra capturar adecuadamente las relaciones presentes en los datos, aunque la regresión lineal se comporta mejor como modelo base.

Esto evidencia la necesidad de utilizar modelos más avanzados o técnicas de optimización para mejorar la capacidad predictiva.

## 14. Validación cruzada

Se aplica validación cruzada para evaluar la estabilidad y capacidad de generalización de los modelos.

Este método permite entrenar y evaluar el modelo en múltiples subconjuntos del dataset, reduciendo el riesgo de sobreajuste.

In [16]:
cv_lr = cross_val_score(lr, X_scaled, y, cv=5, scoring='r2').mean()
cv_knn = cross_val_score(knn, X_scaled, y, cv=5, scoring='r2').mean()

cv_lr, cv_knn

(np.float64(0.003171007645559687), np.float64(-0.19733334454433446))

Los resultados de validación cruzada muestran que el modelo de regresión lineal presenta un R² promedio de 0.003, mientras que el modelo KNN obtiene un R² negativo (-0.197).

Estos resultados son consistentes con la evaluación realizada previamente sobre el conjunto de prueba, lo que indica que los modelos presentan un comportamiento estable.

Sin embargo, ambos modelos muestran un bajo poder explicativo, especialmente KNN, cuyo desempeño es inferior a predecir el valor promedio.

Esto confirma que los modelos simples no logran capturar adecuadamente la complejidad del problema, evidenciando la necesidad de aplicar técnicas más avanzadas.

## 15. Optimización del modelo

Se optimiza el modelo KNN mediante búsqueda de hiperparámetros utilizando GridSearchCV.

El objetivo es encontrar el número óptimo de vecinos (k) que minimice el error y mejore la capacidad predictiva del modelo.

In [17]:
param_grid = {'n_neighbors': [3,5,7,9]}

grid = GridSearchCV(
    KNeighborsRegressor(),
    param_grid,
    cv=5,
    scoring='r2'
)

grid.fit(X_train, y_train)

best_knn = grid.best_estimator_

In [18]:
y_pred_knn_opt = best_knn.predict(X_test)

Se generan las predicciones utilizando el modelo KNN optimizado, aplicándolo sobre el conjunto de datos de prueba.

Esto permite evaluar el desempeño del modelo con los mejores hiperparámetros encontrados mediante GridSearchCV.

In [19]:
res_knn_opt = evaluar(y_test, y_pred_knn_opt)

res_knn_opt

{'MAE': 1292.6454844444445,
 'RMSE': np.float64(1516.2759390026997),
 'R2': -0.10804555418324346}

El modelo KNN optimizado presenta una mejora respecto a la versión inicial, evidenciada en la disminución del MAE y RMSE, así como en una leve mejora del coeficiente R².

Sin embargo, a pesar de esta mejora, el desempeño del modelo sigue siendo bajo, manteniendo un R² negativo, lo que indica que no logra capturar adecuadamente la relación entre las variables.

Esto sugiere que, aunque la optimización de hiperparámetros aporta beneficios, el algoritmo KNN no es el más adecuado para este problema, siendo necesario utilizar modelos más avanzados.

## 16. Modelo Boosting

Se implementa un modelo de Gradient Boosting, el cual es un método ensemble que combina múltiples modelos débiles para generar un modelo más robusto.

Este tipo de modelo es capaz de capturar relaciones no lineales y patrones complejos presentes en los datos, lo que lo hace adecuado para este problema.

In [20]:
gb = GradientBoostingRegressor(random_state=42)

gb.fit(X_train, y_train)

,"loss loss: {'squared_error', 'absolute_error', 'huber', 'quantile'}, default='squared_error'Loss function to be optimized. 'squared_error' refers to the squarederror for regression. 'absolute_error' refers to the absolute error ofregression and is a robust loss function. 'huber' is acombination of the two. 'quantile' allows quantile regression (use`alpha` to specify the quantile).See:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_quantile.py`for an example that demonstrates quantile regression for creatingprediction intervals with `loss='quantile'`.",'squared_error'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'The function to measure the quality of a split. Supported criteria are""friedman_mse"" for the mean squared error with improvement score byFriedman, ""squared_error"" for mean squared error. The default value of""friedman_mse"" is generally the best as it can provide a betterapproximation in some cases... versionadded:: 0.18",'friedman_mse'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft 

In [21]:
y_pred_gb = gb.predict(X_test)

Se generan las predicciones utilizando el modelo Gradient Boosting sobre el conjunto de prueba, permitiendo evaluar su desempeño en datos no vistos.

In [22]:
res_gb = evaluar(y_test, y_pred_gb)

res_gb

{'MAE': 1245.520008803158,
 'RMSE': np.float64(1438.3407893377846),
 'R2': 0.002932129192653954}

El modelo de Gradient Boosting presenta un desempeño muy similar al de la regresión lineal, sin lograr mejoras significativas en las métricas evaluadas.

Si bien este tipo de modelo es capaz de capturar relaciones no lineales, en este caso no logra mejorar la capacidad predictiva, manteniendo un R² cercano a 0.

Esto sugiere que las variables disponibles no contienen suficiente información para explicar la variabilidad del monto de compra, independientemente del modelo utilizado.

## 17. Comparación de modelos

Se comparan los modelos entrenados en base a las métricas obtenidas.

Se observa que la regresión lineal y el modelo de Gradient Boosting presentan resultados prácticamente equivalentes, con menor error en comparación con KNN.

Por otro lado, el modelo KNN, incluso después de la optimización, presenta un desempeño inferior, con valores de R² negativos, lo que indica que no es adecuado para este problema.

En general, ningún modelo logra explicar adecuadamente la variabilidad del dataset, evidenciando una baja relación entre las variables predictoras y la variable objetivo.

## 18. Selección del modelo final

Se selecciona el modelo de regresión lineal como modelo final, ya que presenta el mejor desempeño general en términos de error, siendo además el modelo más simple e interpretable.

Aunque el modelo de Gradient Boosting presenta resultados similares, no logra mejoras significativas que justifiquen su mayor complejidad.

Por lo tanto, se opta por la regresión lineal como la alternativa más adecuada para este problema.

## 19. Conclusión

Se desarrolló un pipeline completo de aprendizaje supervisado, incluyendo exploración, limpieza, modelado, evaluación y optimización.

Los resultados evidenciaron que los modelos utilizados no logran capturar adecuadamente la variabilidad del dataset, lo que sugiere una baja relación entre las variables predictoras y la variable objetivo.

En este contexto, se seleccionó la regresión lineal como modelo final por su simplicidad y desempeño comparable al de modelos más complejos.

Esto demuestra la importancia de evaluar críticamente los resultados y no asumir que modelos más avanzados siempre generan mejores predicciones.